In [1]:
%%sql
SELECT COUNT(*) FROM silver.patient_information

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [8]:
%%sql
CREATE OR REPLACE TABLE gold.procedure_baseline AS
SELECT
    procedure_nm,
    COUNT(*) AS case_count,
    CAST(percentile(or_duration_min, 0.5) AS DECIMAL(10,1)) AS expected_duration_min,
    CAST(AVG(or_duration_min) AS DECIMAL(10,1)) AS mean_duration_min,
    CAST(STDDEV(or_duration_min) AS DECIMAL(10,2)) AS stdev_duration_min,
    CAST(STDDEV(or_duration_min) / AVG(or_duration_min) AS DECIMAL(10,3)) AS cv,
    CAST(COUNT(*) * AVG(or_duration_min) / 60 AS DECIMAL(12,1)) AS or_hours_consumed
FROM silver.patient_information
WHERE in_or IS NOT NULL
  AND out_or IS NOT NULL
  AND or_duration_min BETWEEN 10 AND 720
GROUP BY procedure_nm
HAVING COUNT(*) >= 30

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
%%sql
SELECT COUNT(*) AS procedures, SUM(case_count) AS cases FROM gold.procedure_baseline

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [12]:
drop_cols = [
    "hosp_admsn_time", "hosp_disch_time", "surgery_date",
    "los", "icu_admin_flag", "disch_disp_c",
    "height", "weight", "primary_procedure_nm",
    "in_or_dttm", "out_or_dttm", "an_start_datetime", "an_stop_datetime",
]

s = spark.table("silver.patient_information").drop(*drop_cols)

s.createOrReplaceTempView("patient_information_gold")

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 18, Finished, Available, Finished, False)

In [13]:
%%sql

CREATE OR REPLACE TABLE gold.fact_cases AS

SELECT
    s.*,
    b.expected_duration_min,

    CAST(
        s.or_duration_min - b.expected_duration_min
        AS DECIMAL(10,1)
    ) AS schedule_error_min,

    CAST(
        ABS(s.or_duration_min - b.expected_duration_min)
        AS DECIMAL(10,1)
    ) AS abs_error_min,

    s.or_duration_min - b.expected_duration_min > 30 AS overrun_30,
    s.or_duration_min - b.expected_duration_min > 60 AS overrun_60,
    s.or_duration_min - b.expected_duration_min < -30 AS early_30

FROM patient_information_gold s

JOIN gold.procedure_baseline b
    ON s.procedure_nm = b.procedure_nm

WHERE s.in_or IS NOT NULL
  AND s.out_or IS NOT NULL
  AND s.or_duration_min BETWEEN 10 AND 720;

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
%%sql
SELECT COUNT(*) AS cases, COUNT(DISTINCT procedure_nm) AS procedures
FROM gold.fact_cases

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [15]:
%%sql
SELECT
    COUNT(*) AS cases,
    CAST(AVG(abs_error_min) AS DECIMAL(10,1)) AS mae_min,
    CAST(percentile(abs_error_min, 0.5) AS DECIMAL(10,1)) AS median_abs_error_min,
    CAST(100.0 * AVG(CASE WHEN overrun_30 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_overrun_30,
    CAST(100.0 * AVG(CASE WHEN overrun_60 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_overrun_60,
    CAST(100.0 * AVG(CASE WHEN early_30 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_early_30
FROM gold.fact_cases

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 21, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 6 fields>

In [16]:
%%sql
SELECT
    SUM(abs_error_min) AS total_error_min,
    ROUND(SUM(abs_error_min) / 5.75) AS error_min_per_year,
    ROUND(SUM(abs_error_min) / 5.75 * 35 / 1000000, 1) AS cost_musd_at_35,
    ROUND(SUM(abs_error_min) / 5.75 * 60 / 1000000, 1) AS cost_musd_at_60
FROM gold.fact_cases

StatementMeta(, 67480a8e-4982-43ff-82b0-d6fc285d55b2, 22, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>